In [1]:
!pip install torch_geometric --quiet

# kaggle pytorch version and cuda version:
# pytorch version 2.10.0
# cuda version 12.8
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html --quiet

In [2]:
import torch
from torch import nn, Tensor
import torch.nn.functional as F

from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    classification_report
)

from sklearn.utils.class_weight import compute_class_weight

In [ ]:
RANDOM_STATE = 42

# target variable
TARGET = 'Traffic'
TO_DROP = ['Target', 'Traffic', 'StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'sIpId', 'dIpId']

df = pd.read_csv('../data/wustl_iiot_2021.csv')

In [4]:
df['Dport'].value_counts()

Dport
502      1103533
80         52935
0          11114
8080        7221
1740        4812
          ...   
55127          1
53549          1
63261          1
250            1
39951          1
Name: count, Length: 7781, dtype: int64

In [5]:
df.columns

Index(['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'Mean', 'Sport', 'Dport',
       'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
       'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
       'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt',
       'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max',
       'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes',
       'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct',
       'Traffic', 'Target'],
      dtype='object')

In [6]:
all_ip_addresses = set(df['SrcAddr']).union(set(df['DstAddr']))

ip_addr_mapping = {ip_address: i for i, ip_address in enumerate(all_ip_addresses)}

In [7]:
ip_addr_mapping

{'ff02::1': 0,
 '16274': 1,
 '192.168.0.255': 2,
 '25121': 3,
 '0.0.0.0': 4,
 '192.168.0.10': 5,
 '01:80:c2:00:00:0e': 6,
 '66540': 7,
 '224.0.0.252': 8,
 '5083': 9,
 '82621': 10,
 '5106': 11,
 '5100': 12,
 '17610': 13,
 '4871': 14,
 '82010': 15,
 '255.255.255.255': 16,
 '66330': 17,
 '5078': 18,
 '7103': 19,
 '5111': 20,
 '65771': 21,
 '5095': 22,
 '5096': 23,
 '239.255.255.250': 24,
 '4873': 25,
 '224.0.0.251': 26,
 '4904': 27,
 '66529': 28,
 '5110': 29,
 '45736': 30,
 '65886': 31,
 '66385': 32,
 '30625': 33,
 '66401': 34,
 '66414': 35,
 '192.168.0.2': 36,
 '66544': 37,
 '66359': 38,
 'fe80::e9ed:931f:c2e0:1333': 39,
 'ff02::2': 40,
 '5103': 41,
 '29533': 42,
 '65972': 43,
 '5094': 44,
 '0.0.0.1': 45,
 '17614': 46,
 '4504': 47,
 '4879': 48,
 '4951': 49,
 '6c:b0:ce:22:b0:57': 50,
 '5088': 51,
 '5113': 52,
 '66448': 53,
 '20.1.249.77': 54,
 '75281': 55,
 '66138': 56,
 '30741': 57,
 '99793': 58,
 '5105': 59,
 '5087': 60,
 '14740': 61,
 '6176': 62,
 '72918': 63,
 '66525': 64,
 '28257': 6

In [8]:
df.columns

Index(['StartTime', 'LastTime', 'SrcAddr', 'DstAddr', 'Mean', 'Sport', 'Dport',
       'SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
       'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
       'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt',
       'DIntPkt', 'Proto', 'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max',
       'sDSb', 'sTtl', 'dTtl', 'sIpId', 'dIpId', 'SAppBytes', 'DAppBytes',
       'TotAppByte', 'SynAck', 'RunTime', 'sTos', 'SrcJitAct', 'DstJitAct',
       'Traffic', 'Target'],
      dtype='object')

In [9]:
# private ip addresses start with 192.168
df['is_private_src'] = df['SrcAddr'].apply(lambda x: 1 if x.startswith('192.168') else 0)
df['is_private_dst'] = df['DstAddr'].apply(lambda x: 1 if x.startswith('192.168') else 0)

In [10]:
# encode the ip addresses into node indices
df['SrcAddr'] = df['SrcAddr'].map(ip_addr_mapping)
df['DstAddr'] = df['DstAddr'].map(ip_addr_mapping)

df = df.sort_values(by=['SrcAddr', 'DstAddr'])

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[TARGET]
)

df_train, df_val = train_test_split(
    df_train,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=df_train[TARGET]
)

encoder = LabelEncoder()
df_train[TARGET] = encoder.fit_transform(df_train[TARGET])
df_val[TARGET] = encoder.transform(df_val[TARGET])
df_test[TARGET] = encoder.transform(df_test[TARGET])

In [17]:
continuous_cols = ['SrcPkts', 'DstPkts', 'TotPkts', 'DstBytes', 'SrcBytes', 'TotBytes',
    'SrcLoad', 'DstLoad', 'Load', 'SrcRate', 'DstRate', 'Rate', 'SrcLoss',
    'DstLoss', 'Loss', 'pLoss', 'SrcJitter', 'DstJitter', 'SIntPkt', 'DIntPkt',
    'Dur', 'TcpRtt', 'IdleTime', 'Sum', 'Min', 'Max', 'SAppBytes', 'DAppBytes',
    'TotAppByte', 'RunTime', 'SrcJitAct', 'DstJitAct', 'Sport', 'Dport', 'sTtl', 'dTtl', 'SynAck']

categorical_cols = ['Proto', 'sDSb', 'sTos']

binary_cols = ['is_private_src', 'is_private_dst']

# fit on train only
scaler = StandardScaler()
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

scaler.fit(df_train[continuous_cols])
ohe.fit(df_train[categorical_cols])

ohe_col_names = ohe.get_feature_names_out(categorical_cols)

def transform_df(df, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols):
    df = df.copy()

    # continuous -> standardized, overwrite in place
    df[continuous_cols] = scaler.transform(df[continuous_cols])

    # categorical -> one-hot, new columns appended
    ohe_array = ohe.transform(df[categorical_cols])
    ohe_df = pd.DataFrame(ohe_array, columns=ohe_col_names, index=df.index)
    df = pd.concat([df, ohe_df], axis=1)

    # drop original categorical columns now that they're one-hot encoded
    df = df.drop(columns=categorical_cols)

    return df

df_train = transform_df(df_train, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols)
df_val = transform_df(df_val, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols)
df_test = transform_df(df_test, scaler, ohe, continuous_cols, categorical_cols, ohe_col_names, binary_cols)

In [18]:
df_train

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,sDSb_0,sDSb_4,sDSb_31,sDSb_48,sDSb_51,sTos_0,sTos_16,sTos_126,sTos_192,sTos_207
953646,2019-08-19 14:49:05,2019-08-19 14:49:05,133,36,0,-0.100357,-0.087112,-0.003024,-0.007830,-0.003057,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
234332,2019-08-19 13:07:03,2019-08-19 13:07:07,104,36,4,-1.254293,-0.215079,-0.003193,-0.015634,-0.003452,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1005496,2019-08-19 12:00:35,2019-08-19 12:00:35,133,36,0,0.491434,-0.087112,-0.003137,-0.009781,-0.003227,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
19388,2019-08-19 15:38:16,2019-08-19 15:38:16,133,36,0,0.456749,-0.087112,-0.003137,-0.009781,-0.003227,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
111540,2019-08-19 12:21:12,2019-08-19 12:21:12,133,36,0,-0.147017,-0.087112,-0.003024,-0.007830,-0.003057,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
739715,2019-08-19 14:51:37,2019-08-19 14:51:37,133,36,0,0.483672,-0.087112,-0.003024,-0.007830,-0.003057,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1011632,2019-08-19 15:53:08,2019-08-19 15:53:08,133,36,0,-0.163203,-0.087112,-0.003024,-0.007830,-0.003057,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1082780,2019-08-19 10:27:56,2019-08-19 10:27:56,133,36,0,0.748433,-0.087112,-0.003137,-0.009781,-0.003227,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1075978,2019-08-19 15:37:54,2019-08-19 15:37:54,133,36,0,0.370450,-0.087112,-0.003024,-0.007830,-0.003057,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [19]:
# 0 is backdoor, 1 is CommInj, ...
encoder.classes_

array(['Backdoor', 'CommInj', 'DoS', 'Reconn', 'normal'], dtype=object)

In [20]:
# create edges (in the form of an edge list first) and edge features
edge_list_train = torch.tensor(
    df_train[['SrcAddr', 'DstAddr']].to_numpy(),
    dtype=torch.long
)
edge_features_train = torch.tensor(
    df_train.drop(columns=TO_DROP).to_numpy(),
    dtype=torch.float64
)
y_train = torch.tensor(
    df_train[TARGET].to_numpy(),
    dtype=torch.long
)


edge_list_val = torch.tensor(
    df_val[['SrcAddr', 'DstAddr']].to_numpy(),
    dtype=torch.long
)
edge_features_val = torch.tensor(
    df_val.drop(columns=TO_DROP).to_numpy(),
    dtype=torch.float64
)
y_val = torch.tensor(
    df_val[TARGET].to_numpy(),
    dtype=torch.long
)


edge_list_test = torch.tensor(
    df_test[['SrcAddr', 'DstAddr']].to_numpy(),
    dtype=torch.long
)
edge_features_test = torch.tensor(
    df_test.drop(columns=TO_DROP).to_numpy(),
    dtype=torch.float64
)
y_test = torch.tensor(
    df_test[TARGET].to_numpy(),
    dtype=torch.long
)

# TODO: create node features

# TODO: see if there's a way to include time into the graphs as well


In [21]:
# NOTE: we did not create any node features yet, so use all ones at the moment
edge_features_train = edge_features_train.float()
train_data = Data(
    edge_index=edge_list_train.t().contiguous(),
    edge_attr=edge_features_train.float(),
    edge_label=y_train,
    num_classes=len(y_train.unique()),
    x=torch.ones(
        (len(ip_addr_mapping), 1),
        dtype=torch.float32
    )
)

edge_features_val = edge_features_val.float()
val_data = Data(
    edge_index=edge_list_val.t().contiguous(),
    edge_attr=edge_features_val.float(),
    edge_label=y_val,
    num_classes=len(y_val.unique()),
    x=torch.ones(
        (len(ip_addr_mapping), 1),
        dtype=torch.float32
    )
)

edge_features_test = edge_features_test.float()
test_data = Data(
    edge_index=edge_list_test.t().contiguous(),
    edge_attr=edge_features_test.float(),
    edge_label=y_test,
    num_classes=len(y_test.unique()),
    x=torch.ones(
        (len(ip_addr_mapping), 1),
        dtype=torch.float32
    )
)

train_data.validate(raise_on_error=True)
val_data.validate(raise_on_error=True)
test_data.validate(raise_on_error=True)

True

### Model definition

In [22]:
class GCN(nn.Module):
    def __init__(self, num_edge_features, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels, add_self_loops=True)
        self.conv2 = GCNConv(hidden_channels, hidden_channels, add_self_loops=True)
        self.conv3 = GCNConv(hidden_channels, hidden_channels, add_self_loops=True)

        self.classifier = nn.Linear(hidden_channels * 2 + num_edge_features, out_channels)

        # xavier glorot initialization
        self._init_parameters()

    def forward(self, x: Tensor, edge_index: Tensor, edge_attr: Tensor, edge_label_index: Tensor, target_edge_attr: Tensor) -> Tensor:
        # x: Node feature matrix of shape [num_nodes, in_channels]
        # edge_index: Graph connectivity matrix of shape [2, num_edges]
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        # skip connection applied here
        identity = x
        x = self.conv2(x, edge_index)
        x = F.relu(x + identity)
        x = F.dropout(x, p=0.1, training=self.training)

        # layer 3
        x = self.conv3(x, edge_index)

        # edge classification task
        # get edge embeddings out from the node embeddings
        row, col = edge_label_index
        edge_src_emb = x[row]  # Shape: [num_edges, hidden_channels]
        edge_dst_emb = x[col]  # Shape: [num_edges, hidden_channels]
        edge_features = torch.cat([edge_src_emb, edge_dst_emb, target_edge_attr], dim=-1)

        # classifier
        x = self.classifier(edge_features)

        return x

    def _init_parameters(self):
        # xavier glorot init by default for this method
        self.conv1.reset_parameters()
        self.conv2.reset_parameters()
        self.conv3.reset_parameters()

        nn.init.xavier_normal_(self.classifier.weight)
        if self.classifier.bias is not None:
            nn.init.zeros_(self.classifier.bias)


model = GCN(
    num_edge_features=train_data.num_edge_features,
    in_channels=train_data.num_features,
    hidden_channels=32,
    out_channels=train_data.num_classes
)

In [23]:
train_dataloader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[15, 15],
    batch_size=256,
    edge_label_index=train_data.edge_index,
    edge_label=train_data.edge_label,
    shuffle=True,
)

val_dataloader = LinkNeighborLoader(
    data=val_data,
    num_neighbors=[15, 15],
    batch_size=256,
    edge_label_index=val_data.edge_index,
    edge_label=val_data.edge_label,
    shuffle=True,
)

test_dataloader = LinkNeighborLoader(
    data=test_data,
    num_neighbors=[15, 15],
    batch_size=256,
    edge_label_index=test_data.edge_index,
    edge_label=test_data.edge_label,
    shuffle=True,
)

/usr/local/lib/python3.12/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# set class weights for loss function
class_weights = compute_class_weight(class_weight='balanced', y=y_train.numpy(), classes=np.unique(y_train.numpy()))
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.to(device)

GCN(
  (conv1): GCNConv(1, 32)
  (conv2): GCNConv(32, 32)
  (conv3): GCNConv(32, 32)
  (classifier): Linear(in_features=122, out_features=5, bias=True)
)

In [25]:
def train(model, edge_features, dataloader, loss_fn, optimizer, device, num_classes=5):
    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []
    all_probs = []

    edge_features = edge_features.to(device)

    for batch in dataloader:
        batch = batch.to(device)

        optimizer.zero_grad()

        target_edge_attr = edge_features[batch.input_id]
        # forward pass
        outputs = model(
            batch.x,
            batch.edge_index,
            batch.edge_attr,
            batch.edge_label_index,
            target_edge_attr
        )

        # outputs: [num_target_edges, num_classes]
        # batch.edge_label: [num_target_edges]
        loss = loss_fn(outputs, batch.edge_label)
        # backpropagation
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Probabilities
        probs = torch.softmax(outputs, dim=1)
        # Predicted class
        preds = outputs.argmax(dim=1)

        # Move to CPU for sklearn
        all_probs.append(probs.detach().cpu().numpy())
        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(batch.edge_label.detach().cpu().numpy())

    # Combine all batches
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Average loss
    avg_loss = running_loss / len(dataloader)

    # Classification metrics
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    # Multiclass PR-AUC
    one_hot_labels = np.eye(num_classes)[all_labels]

    pr_auc = average_precision_score(
        one_hot_labels,
        all_probs,
        average="macro"
    )

    return {
        "loss": avg_loss,
        "f1": macro_f1,
        "precision": macro_precision,
        "recall": macro_recall,
        "pr_auc": pr_auc
    }

def evaluate(model, edge_features, dataloader, loss_fn, optimizer, device, num_classes=5):
    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []
    all_probs = []

    edge_features = edge_features.to(device)

    with torch.no_grad():

        for batch in dataloader:
            batch = batch.to(device)

            # Forward pass
            target_edge_attr = edge_features[batch.input_id]
            outputs = model(
                batch.x,
                batch.edge_index,
                batch.edge_attr,
                batch.edge_label_index,
                target_edge_attr
            )

            # Loss
            loss = loss_fn(outputs, batch.edge_label)
            running_loss += loss.item()

            # Probabilities
            probs = torch.softmax(outputs, dim=1)

            # Predicted class
            preds = outputs.argmax(dim=1)

            # Store for metrics
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch.edge_label.cpu().numpy())

    # Combine batches
    all_probs = np.concatenate(all_probs)
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    # Average loss
    avg_loss = running_loss / len(dataloader)

    # Metrics
    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    # Multiclass PR-AUC
    one_hot_labels = np.eye(num_classes)[all_labels]

    pr_auc = average_precision_score(
        one_hot_labels,
        all_probs,
        average="macro"
    )

    return {
        "loss": avg_loss,
        "f1": macro_f1,
        "precision": macro_precision,
        "recall": macro_recall,
        "pr_auc": pr_auc
    }

In [26]:
NUM_EPOCHS = 30

# TODO: maybe can try 1 pass the whole graph in but not sure if it will fit in memory or not
# TODO: add early stopping
# TODO: add saving model after every x epochs, try changing learning rate
# TODO: run notebook again without the preprocessing code once preprocessing is done
# TODO: add plots of confusion matrix, etc. once done training
# TODO: train on both train and val sample for X epochs then evaluate on test data

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = train(model, edge_features_train, train_dataloader, loss_fn, optimizer, device)
    val_metrics = evaluate(model, edge_features_val, val_dataloader, loss_fn, optimizer, device)

    print(f"Epoch {epoch}: ")
    print(
        f"Train loss = {train_metrics['loss']}, "
        f"Train f1: {train_metrics['f1']}, "
        f"Train precision: {train_metrics['precision']}, "
        f"Train recall: {train_metrics['recall']}, "
        f"Train PR-AUC: {train_metrics['pr_auc']}"
    )
    print(
        f"Validation loss = {val_metrics['loss']}, "
        f"Validation f1: {val_metrics['f1']}, "
        f"Validation precision: {val_metrics['precision']}, "
        f"Validation recall: {val_metrics['recall']}, "
        f"Validation PR-AUC: {val_metrics['pr_auc']}"
    )
    print("-" * 100)

Epoch 1: 
Train loss = 0.21360963777468506, Train f1: 0.6176360296179962, Train precision: 0.546149452920891, Train recall: 0.8630390349604348, Train PR-AUC: 0.7396258019971157
Validation loss = 0.08016996500816913, Validation f1: 0.6321363019160959, Validation precision: 0.5830150422478846, Validation recall: 0.9051580817749205, Validation PR-AUC: 0.7914817200059097
----------------------------------------------------------------------------------------------------
Epoch 2: 
Train loss = 0.08196264855301014, Train f1: 0.6514223723625905, Train precision: 0.5993074607858443, Train recall: 0.916479921808255, Train PR-AUC: 0.7975269671701957
Validation loss = 0.057359556743456795, Validation f1: 0.7471132142856481, Validation precision: 0.7300904550047118, Validation recall: 0.9346392472850346, Validation PR-AUC: 0.7997156933029846
----------------------------------------------------------------------------------------------------
Epoch 3: 
Train loss = 0.06658724524060658, Train f1: 0.6

In [16]:
print(train_data.x.dtype)
print(train_data.edge_attr.dtype)
print(train_data.edge_label.dtype)

torch.float32
torch.float32
torch.int64
